JNPA Demurrage Fee Calculator 

Phase 1: Data Cleaning & Preprocessing

Objective: Load the raw digitized dataset, fix format issues, and handle missing values to create a continuous timeline.
Libraries used:
pandas: For data manipulation and dataframe operations.
numpy: For handling numerical operations.


In [25]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")


Libraries imported successfully.


Loading the raw data

In [26]:
# Load the dataset

df = pd.read_excel("../../data/JNPA_Master_Raw.xlsx")


# Check the initial state of data
print(f"Dataset Shape: {df.shape[0]} Rows, {df.shape[1]} Columns")

# Display the last 5 rows to see the missing values (NaN) in recent months
df.tail()

Dataset Shape: 35 Rows, 14 Columns


,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,Exp_Transit_CFS,Exp_Transit_ICD,Exp_Transit_Parking,Total_TEUs_Handled
30,Jul 25,22.0,18.7,63.4,1.9,2.4,112.3,78.1,76.5,91.4,4.8,89.1,1.9,675605
31,Aug 25,26.8,23.7,52.6,2.4,2.6,110.7,78.0,76.3,94.2,5.2,84.2,2.4,716056
32,Sep 25,43.2,37.1,83.6,2.1,2.8,105.0,71.5,69.7,85.5,4.2,92.0,2.1,668404
33,Oct 25,25.3,23.2,45.6,2.1,2.7,111.0,70.1,68.4,84.0,4.3,91.1,2.1,688409
34,Nov 25,28.3,24.9,77.3,2.1,2.9,114.1,68.8,67.0,86.4,4.1,81.7,2.1,668465


2. Date Formatting & Sorting

The 'Month-Year' column is currently a String (text) object (e.g., "Jan 24").
We need to convert this into a Datetime object so Python understands the chronological order.

Method used: pd.to_datetime()

In [27]:
# Convert 'Month-Year' to datetime objects
try:
    df['Date'] = pd.to_datetime(df['Month-Year'], format='%b %y')
except:
    # Fallback if format is slightly different (e.g. full month name)
    print("Standard format failed, attempting inference...")
    df['Date'] = pd.to_datetime(df['Month-Year'], infer_datetime_format=True)

df = df.sort_values(by='Date').reset_index(drop=True)

print(df[['Month-Year', 'Date']].head(3))

  Month-Year       Date
0     Jan 23 2023-01-01
1     Feb 23 2023-02-01
2     Mar 23 2023-03-01


3. Dropping Similar Columns

During data entry, we noticed that Exp_Transit_Parking (Column M) is identical to Parking_Dwell (Column E) in the LDB reports. Keeping both will cause Multicollinearity (confusing the model with duplicate signals).

Method used: df.drop()

In [28]:
# Check if the duplicate column exists before dropping
if 'Exp_Transit_Parking' in df.columns:
    df = df.drop(columns=['Exp_Transit_Parking'])
    print("Dropped duplicate column: Exp_Transit_Parking")
else:
    print("Column already removed or not found.")


print("\nCurrent Columns:", df.columns.tolist())

Dropped duplicate column: Exp_Transit_Parking

Current Columns: ['Month-Year', 'Imp_Dwell_Overall', 'Imp_Dwell_Truck', 'Imp_Dwell_Train', 'Parking_Dwell', 'Imp_Transit_CFS', 'Imp_Transit_ICD', 'Exp_Dwell_Overall', 'Exp_Dwell_Truck', 'Exp_Dwell_Train', 'Exp_Transit_CFS', 'Exp_Transit_ICD', 'Total_TEUs_Handled', 'Date']


4. Handling Missing Values (Interpolation)

We have gaps in the data (example: Jan 2025,etc.).
Instead of filling them with the average (which flattens the trends) or zero (which is false),
we use Linear Interpolation.
Method used: interpolate(method='linear')

In [29]:
# Check missing values before fixing
print("Missing values before interpolation:\n")
print(df.isnull().sum())

numeric_cols = df.select_dtypes(include=[np.number]).columns

# Apply Linear Interpolation
df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both')

df = df.round(2)

print("Interpolation Complete.")

Missing values before interpolation:

Month-Year            0
Imp_Dwell_Overall     1
Imp_Dwell_Truck       0
Imp_Dwell_Train       0
Parking_Dwell         1
Imp_Transit_CFS       0
Imp_Transit_ICD       1
Exp_Dwell_Overall     0
Exp_Dwell_Truck       0
Exp_Dwell_Train       0
Exp_Transit_CFS       0
Exp_Transit_ICD       2
Total_TEUs_Handled    0
Date                  0
dtype: int64
Interpolation Complete.


5.Cross Check and converting to CSV

In [30]:
# Check if missing values are now 0
print(f"Remaining Missing Values: {df.isnull().sum().sum()}")

print(df.loc[df['Month-Year'] == 'Jan 25'])

df.to_csv("../../data/JNPA_Cleaned_Data.csv", index=False)

Remaining Missing Values: 0
   Month-Year  Imp_Dwell_Overall  Imp_Dwell_Truck  Imp_Dwell_Train  \
24     Jan 25              22.05             18.9            100.6   

    Parking_Dwell  Imp_Transit_CFS  Imp_Transit_ICD  Exp_Dwell_Overall  \
24            2.5              2.4            110.0               77.2   

    Exp_Dwell_Truck  Exp_Dwell_Train  Exp_Transit_CFS  Exp_Transit_ICD  \
24             76.0             86.3              5.0             93.7   

    Total_TEUs_Handled       Date  
24              586938 2025-01-01  
